## Notebook 15 — Robustness: Fidelity Map & Output across 3 OSim Checkpoints (Qwen family)

Paper §7.2. Three checkpoints that are architecturally identical but differ
in the stage of human-simulation training (OdysSim, arXiv:2606.14199):

| tag | model | stage |
|---|---|---|
| `base` | `Qwen/Qwen3-8B-Base` | no simulation training yet |
| `mid`  | `cmu-lti/osim-8b-mid` | midtrained on a 10B-token behavioral corpus |
| `osim` | `cmu-lti/osim-8b` | + RL/consolidation (final) |

**Two measurements per checkpoint, the EXACT SAME prompts for all three:**

1. **Map (Part A):** representation extraction over 4 templates × 169 cells
   at every layer — residual, per-head (o_proj input), FFN. Map analysis +
   selection correction are done LOCALLY (`pipeline/09_robustness_cross_model/robustness_map.py`).
2. **Mouth (Part B):** answer-distribution predictions (softmax over letters)
   for broad-coverage questions (random 40/type, seed 42) — for output
   fidelity (RSA of the output RDM vs the survey) per checkpoint.

With both we can answer: does simulation training change the map, the
mouth, both, or nothing at all — and whether the map-vs-mouth dissociation
(finding 11) also exists in another model family.

⚠️ A note on fairness: `osim-8b` was trained with a chat template; here all
three checkpoints are deliberately given identical raw prompts so they stay
comparable. For `osim`/`mid` that is slightly off-distribution — recorded as
a caveat in the paper, not hidden.


## Before running: Kaggle setup

1. **Accelerator: GPU T4 x2** (the 8B fp16 model ≈ 16 GB, needs 2 GPUs).
2. Attach a dataset containing `opinionqa_intersectional.csv` (same as notebooks 07-14).
3. Internet ON (downloads 3 models, ~16 GB each; the cache is cleared
   between models so the disk is big enough).
4. Estimate: extraction ~15-25 min/model + mouth ~15-25 min/model → total
   **~2-2.5 hours**. Results are saved **per model** — if the session dies,
   models already finished are safe and will be skipped on a re-run.


In [ ]:
!pip install -q -U "transformers>=4.51" accelerate scipy tqdm


In [ ]:
import os, gc, glob, ast, shutil
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from scipy.stats import wasserstein_distance
from transformers import AutoModelForCausalLM, AutoTokenizer

print(torch.__version__, torch.cuda.device_count(), "GPU")


In [ ]:
MODELS = [
    ("base", "Qwen/Qwen3-8B-Base"),
    ("mid",  "cmu-lti/osim-8b-mid"),
    ("osim", "cmu-lti/osim-8b"),
]

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv not found.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
MAX_OPTIONS = 6
MIN_COVERAGE_FRAC = 0.6
MIN_COVERAGE_ABS = 10
N_Q_MOUTH = 40         # questions per type for the mouth eval (random among those passing coverage)
BATCH_MOUTH = 16

OUT_DIR = "/kaggle/working/robustness_osim"
os.makedirs(OUT_DIR, exist_ok=True)


## Data & prompt (identical to notebook 09/14 — MANDATORY, to stay comparable with Mistral)

`GROUP_KEYS` is sorted exactly as in notebook 09, so the existing
`group_real_dist.npy` (notebook 09's output) can be reused in the local
analysis — it is not recomputed here.


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

GROUP_KEYS = sorted(df["group_key"].unique().tolist())   # ordering = notebook 09
n_g = len(GROUP_KEYS)
group_meta = {}
for gk in GROUP_KEYS:
    attr_type, group_str = gk.split(" :: ", 1)
    v1, v2 = group_str.split(" | ", 1)
    group_meta[gk] = {"attr_type": attr_type, "v1": v1, "v2": v2}
ALL_TYPES = sorted({m["attr_type"] for m in group_meta.values()})
print(n_g, "cells,", len(ALL_TYPES), "types")

qmeta = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)

# --- 4 identity templates (COPIED EXACTLY from notebook 09) ---
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}

def make_templates(attr_type, v1, v2):
    l1, l2 = ATTR_LABELS[attr_type]
    return [
        f"This survey respondent's {l1} is {v1} and their {l2} is {v2}.",
        f"I am answering this survey. My {l1} is {v1} and my {l2} is {v2}.",
        f"Survey respondent profile:\n- {l1}: {v1}\n- {l2}: {v2}\nThis respondent is about to answer opinion questions.",
        f"Question: What is this survey respondent's {l1} and {l2}?\nAnswer: Their {l1} is {v1} and their {l2} is {v2}.",
    ]

N_TEMPLATES = 4
prompts_by_template = {t: {gk: make_templates(m["attr_type"], m["v1"], m["v2"])[t]
                           for gk, m in group_meta.items()}
                       for t in range(N_TEMPLATES)}
print("T0 example:", prompts_by_template[0][GROUP_KEYS[0]])


In [ ]:
# --- pick questions for the mouth eval: coverage >=60%, then RANDOM 40/type (seed 42) ---
LETTERS = ["A", "B", "C", "D", "E", "F"]
rng_q = np.random.default_rng(RANDOM_SEED)
mouth_plan = {}
sub_ok = df[df["n_opt"] <= MAX_OPTIONS]
for ty in ALL_TYPES:
    sub = sub_ok[sub_ok["attribute"] == ty]
    cells = sorted(sub["group_key"].unique().tolist())
    qpc = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    cnt = {}
    for gk in cells:
        for qk in qpc.get(gk, set()):
            cnt[qk] = cnt.get(qk, 0) + 1
    min_cov = min(max(MIN_COVERAGE_ABS, int(np.ceil(MIN_COVERAGE_FRAC * len(cells)))), len(cells))
    good = sorted([q for q, c in cnt.items() if c >= min_cov])
    idx = rng_q.choice(len(good), size=min(N_Q_MOUTH, len(good)), replace=False)
    picked = sorted(good[i] for i in idx)
    rows = [(gk, qk) for qk in picked for gk in cells if qk in qpc.get(gk, set())]
    mouth_plan[ty] = dict(cells=cells, questions=picked, rows=rows)
    print(f"[{ty}] {len(cells)} cells, questions used {len(picked)}, rows {len(rows)}")
print("total mouth rows per model:", sum(len(p["rows"]) for p in mouth_plan.values()))

def build_mouth_prompt(gk, qk):
    m = group_meta[gk]
    ident = make_templates(m["attr_type"], m["v1"], m["v2"])[0]   # T0
    question, options, _ = qmeta[qk]
    lines = [ident, "", f"Question: {question}"]
    for i, opt in enumerate(options):
        lines.append(f"{LETTERS[i]}) {opt}")
    lines.append("Answer:")
    return "\n".join(lines)


## Loop over the 3 checkpoints: extraction (Part A) + mouth (Part B), saved per model

The hooks are exactly as in notebook 09: the `self_attn.o_proj` input =
per-head activations BEFORE mixing (Qwen3-8B: 36 layers × 32 heads × 128
dim = 4096, exactly the Mistral pattern). If a model's files already exist
in OUT_DIR → skipped (safe resume). The HF cache is cleared between models
so the disk is big enough.


In [ ]:
def run_one_model(tag, model_path):
    done_flag = os.path.join(OUT_DIR, f"DONE_{tag}.txt")
    if os.path.exists(done_flag):
        print(f"[skip] {tag} already done.")
        return

    print(f"=== {tag}: {model_path} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16, device_map="balanced",
        low_cpu_mem_usage=True)
    model.eval()
    NUM_LAYERS = model.config.num_hidden_layers
    NUM_HEADS = model.config.num_attention_heads
    HIDDEN = model.config.hidden_size
    HEAD_DIM = HIDDEN // NUM_HEADS
    print(f"  {NUM_LAYERS} layers, {NUM_HEADS} heads x {HEAD_DIM} dim")

    # answer letters -> token ids (must be 1 token & unique)
    letter_ids = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
    assert len(set(letter_ids)) == len(letter_ids), "letter ids are not unique!"

    # ---------- Part A: extraction ----------
    _capture = {}
    def _oproj_prehook(li):
        def fn(module, args):
            _capture[("head", li)] = args[0][0, -1, :].detach().float().cpu()
        return fn
    def _mlp_hook(li):
        def fn(module, args, output):
            _capture[("mlp", li)] = output[0, -1, :].detach().float().cpu()
        return fn
    handles = []
    for li, layer in enumerate(model.model.layers):
        handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(li)))
        handles.append(layer.mlp.register_forward_hook(_mlp_hook(li)))

    @torch.no_grad()
    def extract_all(prompt):
        _capture.clear()
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model(**inputs, output_hidden_states=True)
        resid = torch.stack(out.hidden_states, dim=0)[:, 0, -1, :].float().cpu().numpy()
        heads = np.stack([_capture[("head", li)].numpy().reshape(NUM_HEADS, HEAD_DIM)
                          for li in range(NUM_LAYERS)])
        mlp = np.stack([_capture[("mlp", li)].numpy() for li in range(NUM_LAYERS)])
        return resid, heads, mlp

    resid_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS + 1, HIDDEN), dtype=np.float16)
    heads_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, NUM_HEADS, HEAD_DIM), dtype=np.float16)
    mlp_all   = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, HIDDEN), dtype=np.float16)
    for t in range(N_TEMPLATES):
        for i, gk in enumerate(tqdm(GROUP_KEYS, desc=f"{tag} extraction T{t}")):
            r, h, m = extract_all(prompts_by_template[t][gk])
            resid_all[t, i], heads_all[t, i], mlp_all[t, i] = r, h, m
        torch.cuda.empty_cache()
    for h_ in handles:
        h_.remove()
    gkeys_arr = np.array(GROUP_KEYS, dtype=object)
    np.savez_compressed(os.path.join(OUT_DIR, f"emb_resid_{tag}.npz"), emb=resid_all, group_keys=gkeys_arr)
    np.savez_compressed(os.path.join(OUT_DIR, f"emb_heads_{tag}.npz"), emb=heads_all, group_keys=gkeys_arr)
    np.savez_compressed(os.path.join(OUT_DIR, f"emb_mlp_{tag}.npz"),   emb=mlp_all,   group_keys=gkeys_arr)
    del resid_all, heads_all, mlp_all
    gc.collect()
    print(f"  [saved] emb_*_{tag}.npz")

    # ---------- Part B: mouth (batched, left-padding) ----------
    tokenizer.padding_side = "left"

    @torch.no_grad()
    def mouth_batch(prompts, n_opt):
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        logits = model(**inputs).logits[:, -1, :]
        selected = logits[:, letter_ids[:n_opt]].float()
        return torch.softmax(selected, dim=1).cpu().numpy()

    rows_out = []
    for ty in ALL_TYPES:
        plan = mouth_plan[ty]
        by_q = {}
        for gk, qk in plan["rows"]:
            by_q.setdefault(qk, []).append(gk)
        for qk, gks in tqdm(by_q.items(), desc=f"{tag} mouth {ty}"):
            n_opt = len(qmeta[qk][2])
            for s in range(0, len(gks), BATCH_MOUTH):
                bg = gks[s:s + BATCH_MOUTH]
                preds = mouth_batch([build_mouth_prompt(gk, qk) for gk in bg], n_opt)
                for b, gk in enumerate(bg):
                    rows_out.append(dict(model=tag, ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                         pred=",".join(f"{x:.6f}" for x in preds[b])))
    pd.DataFrame(rows_out).to_csv(os.path.join(OUT_DIR, f"mouth_preds_{tag}.csv"), index=False)
    print(f"  [saved] mouth_preds_{tag}.csv ({len(rows_out)} rows)")

    # ---------- cleanup ----------
    del model
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
    with open(done_flag, "w") as f:
        f.write("ok")
    print(f"  [done] {tag}\n")


for tag, path in MODELS:
    run_one_model(tag, path)

print("ALL CHECKPOINTS DONE.")
print(sorted(os.listdir(OUT_DIR)))


## Download (MANDATORY) & local analysis

Download the entire contents of `robustness_osim/` →
`results/09_robustness_cross_model/`
(9 `emb_*.npz` files ≈ 600 MB/model + 3 `mouth_preds_*.csv`).

Then run locally:

```
./venv/Scripts/python.exe pipeline/09_robustness_cross_model/robustness_map.py
```

which computes, per checkpoint: the head/residual fidelity map per type
(+ max-stat permutation & held-out template), output fidelity (RSA of the
output RDM vs the survey), and the base → mid → osim comparison.
`group_real_dist` is reused from notebook 09's output (identical cell ordering).
